# Week 9 — One agent, configured from the whole stack

The lecture placed each component of the course system in the stack, in the context of a
model call, and on one axis: fixed in code, or decided by the model. In this session the
components become **settings** of one LangChain agent. Part 1 builds the agent from a
single configuration; every later part changes one setting, asks again, and compares.
Nothing here is a new mechanism: the retrievers are Week 6's, the memory store Week 7's,
the tools Week 8's.

| Part | Setting | Built in |
|---|---|---|
| 1 | The configuration, and the agent built from it | Weeks 1–8 |
| 2 | The model and its provider | Weeks 1–2 |
| 3 | The instruction and examples | Week 2 |
| 4 | Retrieval: a fixed step or a tool | Weeks 3–6 |
| 5 | Conversation history | Week 7 |
| 6 | Memory: written by the application or by the model | Weeks 7–8 |
| 7 | Tools and limits | Week 8 |
| 8 | The configuration of your own project | — |

`week9/agent_stack.py` holds the components: Week 8's tools, unchanged, and two pieces of
middleware that Parts 4 and 6 explain. Read it when Part 1 first uses it.

**Backend.** As in Weeks 7 and 8, the model runs on **OpenRouter**: `OPENROUTER_API_KEY`
must be in `.env` at the repository root. Free models allow **50 requests a day** per
account. One pass through this notebook makes about 25. After Part 1 every part starts
from the same configuration, so a part can be run on its own; running a cell again makes
its requests again. The retrievers and the memory embed with your Gemini key, and the
exchange-rate tool needs an internet connection. Part 2 also runs the agent on Google's
Gemini API, with the same Gemini key; its free tier allows 20 requests a day per model.

**Install.** This week adds `langchain-google-genai`, LangChain's chat model for Gemini.
From the repository root, with your environment active, run
`pip install -r requirements.txt`, then restart the kernel.

In [93]:
# Run this notebook from any folder. The components live in week9/; they reuse the
# retrievers, the memory store, the agent helpers, and the modules those need.
import sys, pathlib
_root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (p / "requirements.txt").exists()), pathlib.Path.cwd().parent)
for _d in ("common", "week2", "week3", "week4", "week5", "week6", "week7", "week8", "week9"):
    sys.path.insert(0, str(_root / _d))
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

---
## Setup — the course graph from Week 6

This notebook reads the graph that Week 6 built and does **not** rebuild it: rebuilding
empties the database, and your Lab 2 graph with it. If the check below fails, run
`python week6/build_graph.py` from the repository root, then re-run your Lab 2
`ingest.py`.

Neo4j must be running: start Docker Desktop first, then `docker compose up -d` from the
repository root.

In [94]:
from embedding_client import EmbeddingClient
from graph_db import connect
from retrievers import read
from vector_check import require_same_model

driver = connect()
emb = EmbeddingClient()
require_same_model(driver, emb)
print(read(driver, "MATCH (e:Entity) RETURN count(e) AS entities")[0],
      read(driver, "MATCH (c:Chunk) RETURN count(c) AS chunks")[0])

{'entities': 372} {'chunks': 183}


You should see 372 entities and 183 chunks, and no error from the embedding-model check.

---
## Part 1 — The configuration

The components, each built in an earlier week. `TOOLS` holds Week 8's tools by name;
`RETRIEVERS` the retrievers of Weeks 3 and 6, each a function from a question to a list of
sources; `store` is Week 7's memory store, in a collection of this notebook's own.

In [76]:
from agent_stack import course_tools, memory_tools, search_tool, context, extract_after, as_result
from memory import MemoryStore
from retrievers import vector, by_name, hybrid

TOOLS = course_tools(driver, emb)                            # the tools, by name
RETRIEVERS = {"vector": lambda q: vector(driver, emb, q),    # the nearest passages
              "by_name": lambda q: by_name(driver, q),       # facts about the names in the question
              "hybrid": lambda q: hybrid(driver, emb, q)}    # passages, and facts about what they mention
store = MemoryStore(name="week9_memory", reset=True)         # the memory store, emptied here

print("tools:", ", ".join(TOOLS))
print("retrievers:", ", ".join(RETRIEVERS))

tools: exchange_rate, calculate, facts_by_name, documents_mentioning
retrievers: vector, by_name, hybrid


**The chat model.** LangChain gives one interface to many providers. `chat_model` returns
it for the two used here: `ChatOpenAI` pointed at OpenRouter, the main provider of this
notebook, or `ChatGoogleGenerativeAI`, from the package `langchain-google-genai`, for
Google's Gemini API. Without a model name it takes the provider's pinned model from
`common/llm_client.py`. See LangChain's
[Models](https://docs.langchain.com/oss/python/langchain/models) and
[ChatGoogleGenerativeAI](https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai)
pages.

In [95]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from agent_kit import COUNTER, openrouter_model
from llm_client import LLMClient


def chat_model(provider: str = "openrouter", model: str | None = None):
    # One LangChain interface, two providers; no model given means the provider's pinned one.
    model = model or LLMClient(provider=provider).model
    if provider == "openrouter":
        return openrouter_model(model)                   # ChatOpenAI at OpenRouter's address
    if provider == "gemini":
        return ChatGoogleGenerativeAI(model=model, api_key=os.environ["GEMINI_API_KEY"],
                                      timeout=120, max_retries=6, callbacks=[COUNTER])
    raise ValueError(f"unknown provider {provider!r}: use 'openrouter' or 'gemini'")


print("openrouter:", chat_model("openrouter").model_name)
print("gemini:    ", chat_model("gemini").model)

openrouter: nvidia/nemotron-3-ultra-550b-a55b:free
gemini:     gemini-3.5-flash


The configuration. Every field is one component:

In [78]:
from dataclasses import dataclass, fields, replace

INSTRUCTION = ("You are the assistant of the Modern AI Systems course. Answer only from the "
               "conversation, the SOURCES and the results of the tools you call, and cite "
               "sources as [n]. Call a tool only when the question needs it. If none of "
               "these contains the answer, reply: not stated.")


@dataclass
class Config:
    provider: str = "openrouter"                   # "openrouter" or "gemini"
    model: str | None = None                       # None: the provider's pinned model
    instruction: str = INSTRUCTION                 # the role and the standing rules
    examples: tuple = ()                           # (question, answer) demonstrations
    retriever: str = "hybrid"                      # "vector", "by_name" or "hybrid"
    retrieval_mode: str = "tool"                   # "fixed": before every answer; "tool": the model decides
    memory: str = "none"                           # "none", "fixed" or "tools"
    summarise_after: int | None = None             # summarise older messages beyond this many
    tools: tuple = ("exchange_rate", "calculate")  # names from TOOLS
    max_searches: int = 2                          # calls to search_course in one run
    max_model_calls: int = 6                       # model calls in one run
    tool_errors_to_model: bool = True              # an exception in a tool goes back to the model

`build_agent` turns each setting into an argument of `create_agent` or into middleware,
code that runs around each model call or tool call (Week 8):

| Setting | Becomes | Documentation |
|---|---|---|
| `provider`, `model` | the chat model, from `chat_model` | [Models](https://docs.langchain.com/oss/python/langchain/models) |
| `instruction`, `examples` | the system prompt, which `context` builds before every model call | [Dynamic prompt](https://docs.langchain.com/oss/python/langchain/middleware/custom#dynamic-prompt) |
| `retriever`, `retrieval_mode` | sources in the system prompt (`"fixed"`), or the tool `search_course` (`"tool"`) | Part 4 |
| `memory` | recalled facts in the system prompt and `extract_after` (`"fixed"`), or the tools `recall_memory` and `remember` (`"tools"`) | Part 6 |
| `summarise_after` | `SummarizationMiddleware` | [Summarization](https://docs.langchain.com/oss/python/langchain/middleware/built-in#summarization) |
| `tools` | Week 8's tools | [Tools](https://docs.langchain.com/oss/python/langchain/tools) |
| `max_searches`, `max_model_calls` | `ToolCallLimitMiddleware`, `ModelCallLimitMiddleware` | [Tool call limit](https://docs.langchain.com/oss/python/langchain/middleware/built-in#tool-call-limit), [Model call limit](https://docs.langchain.com/oss/python/langchain/middleware/built-in#model-call-limit) |
| `tool_errors_to_model` | `ToolErrorMiddleware` | [Tool error](https://docs.langchain.com/oss/python/langchain/middleware/built-in#tool-error) |

In [96]:
from langchain.agents import create_agent
from langchain.agents.middleware import (ModelCallLimitMiddleware, SummarizationMiddleware,
                                         ToolCallLimitMiddleware, ToolErrorMiddleware)


def build_agent(c: Config):
    # Every setting becomes one argument of create_agent, or one piece of middleware.
    model = chat_model(c.provider, c.model)
    retrieve = RETRIEVERS[c.retriever]

    tools = [TOOLS[name] for name in c.tools]
    if c.retrieval_mode == "tool":
        tools.append(search_tool(retrieve))              # the model decides to retrieve
    if c.memory == "tools":
        tools += memory_tools(store)                     # the model reads and writes memory

    middleware = [context(c.instruction, c.examples,     # the system prompt, on every call
                          retrieve=retrieve if c.retrieval_mode == "fixed" else None,
                          store=store if c.memory == "fixed" else None)]
    if c.memory == "fixed":
        middleware.append(extract_after(model, store))   # the application writes memory
    if c.summarise_after:
        middleware.append(SummarizationMiddleware(model, trigger=("messages", c.summarise_after),
                                                  keep=("messages", 2)))
    if c.tool_errors_to_model:
        middleware.append(ToolErrorMiddleware(on_error=as_result))
    middleware += [ToolCallLimitMiddleware(tool_name="search_course", run_limit=c.max_searches),
                   ModelCallLimitMiddleware(run_limit=c.max_model_calls, exit_behavior="end")]

    return create_agent(model, tools=tools, middleware=middleware)

The default configuration is close to Week 8's agent: retrieval is a tool, there is no
memory, and the history is kept whole. Build it and ask a course question. `ask` is Week
8's: it prints the turn's steps, the answer, and the model calls; it now also prints the
time, the model that wrote the answer, and the length of the history that the next
question would carry.

In [97]:
from agent_stack import ask

QUESTION = "What distance does Chroma use by default?"      # asked again below

config = Config()
for f in fields(config):
    print(f"{f.name:21} {str(getattr(config, f.name))[:90]}")
print()
history = ask(build_agent(config), QUESTION)

provider              openrouter
model                 None
instruction           You are the assistant of the Modern AI Systems course. Answer only from the conversation, 
examples              ()
retriever             hybrid
retrieval_mode        tool
memory                none
summarise_after       None
tools                 ('exchange_rate', 'calculate')
max_searches          2
max_model_calls       6
tool_errors_to_model  True

user        What distance does Chroma use by default?
assistant   calls search_course {"query": "Chroma default distance metric"}
tool        search_course → this one to the taxonomy; the next slide unpacks HNSW, and the one after takes the middle row in detail. Give a decision heuristic rather than a ranking: start from the data platform the project is alread … (4,060 characters)
assistant   calls search_course {"query": "Chroma default distance cosine similarity space"}
tool        search_course → notes: ≈4 min. Two points to land, and the second is the o

Read the steps as in Week 8: the model chose to search, wrote the query, and answered from
the result.

**Exercise 1.** For each field of `Config`, name the layer of the lecture's stack it
belongs to, and the part of the context it fills, if any.

---
## Part 2 — The model and its provider

`provider` and `model` are two settings. Nothing else depends on them: the tools, the
retrievers and the memory are not tied to a model. First, the same question with a second
free model on OpenRouter. `replace` copies the configuration with one field changed; `config` itself is
not modified.

In [98]:
ALTERNATIVE = "nvidia/nemotron-3.5-super-vl:free"

history = ask(build_agent(replace(config, model=ALTERNATIVE)), QUESTION)

user        What distance does Chroma use by default?
assistant   calls search_course {"query": "Chroma default distance metric"}
tool        search_course → this one to the taxonomy; the next slide unpacks HNSW, and the one after takes the middle row in detail. Give a decision heuristic rather than a ranking: start from the data platform the project is alread … (4,060 characters)
assistant   calls search_course {"query": "Chroma hnsw:space default distance L2 cosine"}
tool        search_course → this one to the taxonomy; the next slide unpacks HNSW, and the one after takes the middle row in detail. Give a decision heuristic rather than a ranking: start from the data platform the project is alread … (4,363 characters)

──── answer ───────────────────────────────────────────────────────────
Chroma's unconfigured default is **squared Euclidean distance** (L2), rather than the
cosine similarity that this course teaches. [1]
─────────────────────────────────────────────────────────────────

Compare the two runs: the tool calls, the answer, the number of model calls and the time.
A model is chosen for the questions the system must answer, and for its speed and its
limits, not for a single answer.

**Another provider.** The same agent on Google's Gemini API: only `provider` changes, and
`chat_model` returns `ChatGoogleGenerativeAI` with the pinned Gemini model. The tools, the
retrievers, the memory and the limits are unchanged.

In [99]:
on_gemini = replace(config, provider="gemini")          # the pinned Gemini model

history = ask(build_agent(on_gemini), QUESTION)

user        What distance does Chroma use by default?
assistant   calls search_course {"query": "Chroma default distance"}
tool        search_course → this one to the taxonomy; the next slide unpacks HNSW, and the one after takes the middle row in detail. Give a decision heuristic rather than a ranking: start from the data platform the project is alread … (4,093 characters)
assistant   calls search_course {"query": "Chroma default distance metric space cosine l2"}
tool        search_course → notes: ≈4 min. Two points to land, and the second is the one that changes a student's design. First: capability does not follow from a data type. MySQL Community can store a vector and parse one from a st … (4,095 characters)
assistant   calls search_course {"query": "hnsw:space"}
tool        search_course → Tool call limit exceeded. Do not call 'search_course' again.

──── answer ───────────────────────────────────────────────────────────
not stated.
───────────────────────────────────────────────

Compare the three runs. *answered by* gives the model the provider reports, which may
differ from the name requested. The Gemini free tier allows 20 requests a day per model,
against OpenRouter's 50 a day across all free models, so a run that makes four model calls
costs a fifth of a day's Gemini budget.

**Exercise 2.** On [OpenRouter's model list](https://openrouter.ai/models), find another
free model that supports tools, set it as `model`, and ask `QUESTION` again. (About three
requests.)

---
## Part 3 — The instruction and examples

`context`, in `agent_stack.py`, builds the system prompt before every model call: the
instruction, then the examples, then anything the application looked up itself (Parts 4
and 6). It is Week 2's context template, assembled by code. Here two rules are added to
the instruction, and one example of the answer expected. `show_context` prints the system
prompt the model will receive, with no model call:

In [106]:
from agent_stack import show_context

RULES = " Answer in Ukrainian, in at most two sentences."
EXAMPLE = ("Which database does the course use for the knowledge graph?",
           "Для графа знань курс використовує Neo4j [1].")

styled = replace(config, instruction=INSTRUCTION + RULES, examples=(EXAMPLE,))

QUESTION = "How many hryvnias is 250 euros at today's NBU rate?"

show_context(QUESTION, styled.instruction, styled.examples)
print()
history = ask(build_agent(styled), QUESTION)

You are the assistant of the Modern AI Systems course. Answer only from the conversation, the SOURCES and the results of the tools you call, and cite  … (314 characters)

EXAMPLES
Question: Which database does the course use for the knowledge graph?
Answer: Для графа знань курс використовує Neo4j [1].

user        How many hryvnias is 250 euros at today's NBU rate?
assistant   calls exchange_rate {"currency": "EUR"}
tool        exchange_rate → 1 EUR = 51.0527 UAH, set by the NBU for 03.11.2026
assistant   calls calculate {"b": 51.0527, "op": "*", "a": 250}
tool        calculate → 12763.175000000001

──── answer ───────────────────────────────────────────────────────────
250 євро за офіційним курсом НБУ на сьогодні дорівнює 12 763,18 гривень (за курсом 51,0527
грн/євро).
───────────────────────────────────────────────────────────────────────
3 model calls in this turn (13 s), 169 since the kernel started
answered by: nvidia/nemotron-3-ultra-550b-a55b:free
history: 6 messages, about 178 

Did the answer follow both rules, and the form of the example? The instruction is the
only component that changes the agent through the prompt alone, and the model may not
follow it. Week 8 found such a rule: the instruction asked for at most two searches, and
the model kept searching until the code refused. A rule that must hold every time belongs
in code, as `max_searches` is.

**Exercise 3.** Add a rule of your own to the instruction, for example on how sources are
cited, and ask again. Then decide, for each rule in `INSTRUCTION`, whether the system could
enforce it in code instead.

---
## Part 4 — Retrieval: a fixed step or a tool

Until Week 8 the application retrieved before every answer. Week 8 gave the retriever to
the model as a tool. `retrieval_mode` chooses between the two:

| `retrieval_mode` | Who decides to retrieve | The query | Where the sources go |
|---|---|---|---|
| `"fixed"` | the application, before every answer | the question as written | the system prompt, under SOURCES |
| `"tool"` | the model | written by the model | the history, as a tool's result |

With `"fixed"`, `context` runs the retriever on the latest question and places the
numbered sources in the system prompt. What the model will receive for `QUESTION`:

In [101]:
fixed = replace(config, retrieval_mode="fixed")

show_context(QUESTION, fixed.instruction, retrieve=RETRIEVERS[fixed.retriever])

You are the assistant of the Modern AI Systems course. Answer only from the conversation, the SOURCES and the results of the tools you call, and cite  … (267 characters)

SOURCES
[1] this one to the taxonomy; the next slide unpacks HNSW, and the one after takes the middle row in detail. Give a decision heuristic rather than a r … (1,205 characters)
[2] notes: ≈4 min. Two points to land, and the second is the one that changes a student's design. First: capability does not follow from a data type.  … (1,195 characters)
[3] since most build the same graph index over the same distance metrics. Point at the line students already have in `vector_store.py`: `hnsw:space` i … (1,316 characters)
[4] Cosine distance builds on Cosine similarity. (stated in week-03-lecture-plan.md#3)
[5] Cosine distance uses Neo4j. (stated in week-03-lecture.md#16)
[6] Embedding uses Chroma. (stated in week-03-practical-plan.md#6)
[7] Ingest uses Chroma. (stated in week-03-practical-plan.md#6, week-03-practical.md#

Now ask, with the same sources in place:

In [102]:
history = ask(build_agent(fixed), QUESTION)

user        What distance does Chroma use by default?

──── answer ───────────────────────────────────────────────────────────
Chroma's unconfigured default is **squared Euclidean** distance, not cosine similarity
[1].
───────────────────────────────────────────────────────────────────────
1 model call in this turn (3 s), 161 since the kernel started
answered by: nvidia/nemotron-3-ultra-550b-a55b:free
history: 2 messages, about 43 tokens


Compare with Part 1: the same question, fewer model calls, and no query written by the
model. The sources do not appear among the steps, because they are in the system prompt,
not in the history.

A fixed step runs for every question, whether it needs retrieval or not:

In [105]:
show_context("How many hryvnias is 250 euros at today's NBU rate?",
             fixed.instruction, retrieve=RETRIEVERS[fixed.retriever])

QUESTION = "How many hryvnias is 250 euros at today's NBU rate?"

history = ask(build_agent(fixed), QUESTION)

You are the assistant of the Modern AI Systems course. Answer only from the conversation, the SOURCES and the results of the tools you call, and cite  … (267 characters)

SOURCES
[1] produces. The section exists so that the Ukrainian cost premium stops being a number on a slide and becomes a property of their project, which mat … (1,269 characters)
[2] a language count, but the course's own measurements run on Ukrainian throughout and retrieve correctly, so treat it as multilingual on evidence ra … (1,299 characters)
[3] # Week 2 · Practical — Assembling a context, one part at a time **Session:** Week 2, session 2 of 2 (practical) **Duration:** 90 minutes **Syllabu … (1,354 characters)
[4] Building a good context builds on Tokens. (stated in week-02-lecture-plan.md#1)
[5] Chain-of-Thought Prompting builds on Tokens. (stated in week-02-lecture.md#6)
[6] How reasoning models emerged builds on Tokens. (stated in week-02-lecture-plan.md#1)
[7] Tokens builds on Byte-pair encoding. (stated i

The application retrieved passages about the course for a question about a currency: an
embedding call, and their tokens on every model call of the run, for nothing. A follow-up
such as Week 7's *"How do we change it?"* is also retrieved as written, unless the
application rewrites it first.

**Exercise 4.** With `retrieval_mode="fixed"`, set `retriever` to `"vector"` and then to
`"by_name"`, and run `show_context` for `QUESTION` (no model call). Which retriever's
sources answer it? For your own project: which questions always need retrieval, and which
retriever would you fix in code?

---
## Part 5 — Conversation history

The history is every earlier message, sent again with every call (Week 7). `ask` prints
its length after each turn. `summarise_after` adds `SummarizationMiddleware`: once the
history reaches that many messages, one model call summarises the older ones, the
summary replaces them, and the last two messages are kept whole.

The conversation below uses fixed retrieval and no tools, so each turn is one model call
and adds two messages to the history; the summary is one call more. Its last question can
be answered only from the history.

In [107]:
short = replace(config, retrieval_mode="fixed", tools=(), summarise_after=6)
agent = build_agent(short)

history = ask(agent, "What distance does Chroma use by default?")
history = ask(agent, "Why is a document split into chunks before it is embedded?", history)
history = ask(agent, "What does the grounded instruction tell the model to reply when the "
                     "sources do not answer?", history)
history = ask(agent, "What was my first question, and what was your answer?", history)

user        What distance does Chroma use by default?

──── answer ───────────────────────────────────────────────────────────
Chroma's unconfigured default is **squared Euclidean** distance, not cosine similarity.
[1]
───────────────────────────────────────────────────────────────────────
1 model call in this turn (6 s), 170 since the kernel started
answered by: nvidia/nemotron-3-ultra-550b-a55b:free
history: 2 messages, about 43 tokens
user        Why is a document split into chunks before it is embedded?

──── answer ───────────────────────────────────────────────────────────
A single embedding for an entire document can only say “this document, broadly, is
relevant.” It cannot indicate which paragraph or sentence actually answers the question,
because a 40-page file and a one-sentence answer would compress to the **same-sized
description**. Chunking lets retrieval point at the specific part that answers the
question rather than confirming the whole document is somewhat relevant. [1

In the fourth turn the history reached six messages, and the summary is printed before
that turn's steps. Compare the history after the third and the fourth turn, in messages
and in tokens. LangChain's summary follows a fixed template, with sections for the
session's intent, a summary, artefacts and next steps, so on a short conversation it can
be longer than the turns it replaces. A summary costs a model call when it is made and
saves tokens on later calls only once the history is long; what it leaves out is lost to
the rest of the conversation.

Read the summary: does it contain the first question and its answer? Then read the last
answer. The summary is one more message in the history; did the model answer from it, or
take the first question it sees after the summary for the first of the conversation?

**Exercise 5.** Run the conversation again with `summarise_after=None`, and compare the
length of the history and the last answer. (Four requests.) LangChain's
[`ContextEditingMiddleware`](https://docs.langchain.com/oss/python/langchain/middleware/built-in#context-editing)
removes old tool results instead of summarising: Week 7's advice to drop the sources of
earlier turns first.

---
## Part 6 — Memory: written by the application or by the model

Week 7's application wrote memory; Week 8's model wrote it through a tool. `memory`
chooses:

| `memory` | Written | Recalled |
|---|---|---|
| `"fixed"` | after every turn: one model call extracts facts from the user's message, and the code stores them (`extract_after`) | before every model call: the three facts nearest the question, in the system prompt |
| `"tools"` | when the model calls `remember` | when the model calls `recall_memory` |

Both use Week 7's `MemoryStore`, in this notebook's collection `week9_memory`; your Week 7
and Week 8 memories are not touched.

**Written by the application.** First the user tells the agent something about themselves.
Nothing is asked; the point is what the application stores afterwards:

In [108]:
BACKGROUND = ("Some background about me: I missed the Week 3 lecture, and I prefer short "
              "answers of two or three sentences.")
QUESTIONS = ["What did I miss in the lecture I skipped?",
             "Why is a document split into chunks before it is embedded?"]

store = MemoryStore(name="week9_memory", reset=True)        # empty
by_app = replace(config, memory="fixed")

history = ask(build_agent(by_app), BACKGROUND)

memory: 2 facts extracted and stored
  536d80f1  The user missed the Week 3 lecture.
  a1215dfc  The user prefers short answers of two or three sentences.
user        Some background about me: I missed the Week 3 lecture, and I prefer short answers of two or three sentences.

──── answer ───────────────────────────────────────────────────────────
Got it — I’ll keep my answers to 2–3 sentences. What would you like to know about Week 3?
───────────────────────────────────────────────────────────────────────
2 model calls in this turn (24 s), 176 since the kernel started
answered by: nvidia/nemotron-3-ultra-550b-a55b:free
history: 2 messages, about 59 tokens


The lines beginning `memory:` were printed by `extract_after`, which ran after the answer
was written. Now a new conversation: the questions are asked with no history, so whatever
the agent knows about the user comes from memory. What the system prompt holds about the
user before the first question:

In [109]:
show_context(QUESTIONS[0], by_app.instruction, store=store)
print()
agent = build_agent(by_app)
history = ask(agent, QUESTIONS[0])                         # a new conversation: no history
history = ask(agent, QUESTIONS[1], history)

You are the assistant of the Modern AI Systems course. Answer only from the conversation, the SOURCES and the results of the tools you call, and cite  … (267 characters)

WHAT YOU KNOW ABOUT THE USER
- The user missed the Week 3 lecture.
- The user prefers short answers of two or three sentences.

memory: 0 facts extracted and stored
user        What did I miss in the lecture I skipped?
assistant   calls search_course {"query": "Week 3 lecture"}
tool        search_course → # Week 3 · Lecture — Embeddings, semantic search, and RAG **Session:** Week 3, session 1 of 2 (lecture) **Duration:** 90 minutes **Syllabus reference:** `Modern_AI_Systems_Syllabus_v2.md` §6 Week 3 **Guid … (5,488 characters)
assistant   calls search_course {"query": "Week 3 lecture embeddings semantic search RAG"}
tool        search_course → # Week 3 · Lecture — Embeddings, semantic search, and RAG **Session:** Week 3, session 1 of 2 (lecture) **Duration:** 90 minutes **Syllabus reference:** `Modern_AI_Systems_Sylla

**Written by the model.** The same message and the same questions, with the memory tools.
The instruction gains Week 8's rule for using them.

In [110]:
MEMORY_RULE = ("When the user's message tells you something about themselves that will "
               "matter in a later conversation, first store each such fact with remember. "
               "When an answer depends on the user, first recall what is stored. ")

store = MemoryStore(name="week9_memory", reset=True)        # empty again
by_model = replace(config, memory="tools", instruction=MEMORY_RULE + INSTRUCTION)
agent = build_agent(by_model)

history = ask(agent, BACKGROUND)
print()
print(f"{len(store.all())} stored:")
for f in store.all():
    print(f"  {f['id']}  {f['fact']}")
print()
history = ask(agent, QUESTIONS[0])                         # a new conversation: no history
history = ask(agent, QUESTIONS[1], history)

user        Some background about me: I missed the Week 3 lecture, and I prefer short answers of two or three sentences.
assistant   calls remember {"fact": "User missed the Week 3 lecture and prefers short answers of two or three sentences."}
tool        remember → Stored as 8b605cf9.

──── answer ───────────────────────────────────────────────────────────
Noted — I'll keep my replies brief.
───────────────────────────────────────────────────────────────────────
2 model calls in this turn (4 s), 185 since the kernel started
answered by: nvidia/nemotron-3-ultra-550b-a55b:free
history: 4 messages, about 121 tokens

1 stored:
  8b605cf9  User missed the Week 3 lecture and prefers short answers of two or three sentences.

user        What did I miss in the lecture I skipped?
assistant   calls search_course {"query": "lecture content topics covered"}
tool        search_course → Notes | |---|---|---| | 0–10 | **Framing.** The central question; "we are not building another ChatGPT, we are bu

Compare the two positions: which facts were stored, and by whom; whether the answers used
what the user had said, the lecture they missed and the length they prefer; and how many
model calls each turn made.
With `"fixed"`, the extraction runs after every turn, the questions included: read the
`memory:` lines after them. With `"tools"`, the model decides when to write and when to
read: did it call `remember` after the background, and `recall_memory` before answering?

With `"fixed"`, every turn pays for an extraction call, and the code decides what is
written; with `"tools"`, the model decides, and pays only when it chooses to.

**Exercise 6.** For your own project, choose a position for memory. What would you check
before a fact is written, and how would a user correct a wrong one?

---
## Part 7 — Tools and limits

`tools` lists the tools the model may call, beside those that `retrieval_mode` and `memory`
add. Each tool's definition is sent with every model call, whether it is used or not. The
limits are set in code, not in the instruction: `max_searches` refuses searches beyond a
number, and `max_model_calls` stops the run. A run that reaches the limit on model calls:

In [ ]:
tight = replace(config, max_model_calls=3)

history = ask(build_agent(tight), "How many hryvnias is 250 euros at today's NBU rate?")

user        How many hryvnias is 250 euros at today's NBU rate?
assistant   calls exchange_rate {"currency": "EUR"}
tool        exchange_rate → 1 EUR = 51.0527 UAH, set by the NBU for 03.11.2026
assistant   calls calculate {"op": "*", "a": 250, "b": 51.0527}
tool        calculate → 12763.175000000001

──── no answer ────────────────────────────────────────────────────────
The run was stopped at the limit on model calls before the model answered.
Model call limits exceeded: run limit (2/2)
───────────────────────────────────────────────────────────────────────
2 model calls in this turn (7 s), 191 since the kernel started
history: 6 messages, about 163 tokens


The question needs three model calls, and the run was stopped after two, unanswered. Set
the limit from the number of steps your questions need, with room to spare.

**Exercise 7.** Add `"facts_by_name"` and `"documents_mentioning"` to `tools`, and ask
*"Which course documents mention Chroma?"* (About two requests.) Then mark every tool of
the agent as reading or writing. LangChain's
[`HumanInTheLoopMiddleware`](https://docs.langchain.com/oss/python/langchain/middleware/built-in#human-in-the-loop)
pauses the run before a chosen tool is called, so that a person can approve, edit or
reject the call. Which of your project's writes would you put behind it?

---
## Part 8 — The configuration of your own project

**Exercise 8.** Write the configuration of your own project's agent. Your retrievers and
tools are not those of `RETRIEVERS` and `TOOLS`, so this cell is a design, not an agent
to run. Beside each setting, state in a comment whether it is fixed in code or decided by
the model, and why.

In [92]:
my_project = Config(
    provider="openrouter",         # which provider, and why
    model=None,                    # which model: None for the provider's pinned one
    instruction="...",             # the role and the standing rules of your assistant
    examples=(),                   # a demonstration, if the form of the answer matters
    retriever="hybrid",            # which of your retrievers?
    retrieval_mode="fixed",        # fixed or tool: does every question need it?
    memory="none",                 # none, fixed or tools: who writes it, and what is stored?
    summarise_after=None,          # how long do your conversations become?
    tools=("calculate",),          # which tools, and which of them write?
    max_searches=2,
    max_model_calls=6,
)

This configuration is the starting point of Lab 3, due at the end of Week 10: an agent
with instructions, at least two tools and persistent memory, reusing your RAG and Neo4j
components.